In [ ]:
import json
from collections import Counter
from datetime import datetime, timedelta
from itertools import product, zip_longest
from pathlib import Path
from io import StringIO

import numpy as np
import pandas as pd
import plotly.graph_objects as goa
import regex
import requests
import yaml
from plotly.colors import qualitative, sample_colorscale
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
from datetime import datetime
import pytz
from src.utils import (
    guardarExcel,
    guardarExcelMulti
)

from datetime import timedelta

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.utils import (
    isEmpty,
    loadEstaciones,
    loadLocalizaciones,
    localizeFecha,
    parallelizeFunction,
    rellenarId,
    removeDoubleQuotes,
    splitDataframe,
)
from src.api.api import GraylogAPIProcessor
from src.utils.util import loadEstaciones
from src.api.api import GraylogAPIProcessor
from datetime import timedelta
import csv
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from src.api import getHistoricoMOW
from src.utils import (
    formatTimedelta,
    guardarExcel,
    isValidCode,
    rellenarId,
    time2localtime,
)
from src.processor import SitraProcessor
from src.utils.util import (
    loadEstaciones,
    loadEstacionSinCTC
)
import sys


In [ ]:
# graylock = GraylogAPIProcessor()
# today_str = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
# fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra")/ f"{today_str}_Sitra.txt"
# response = graylock.saveResponse(fname,"18-09-2025","19-09-2025",[],"sitra")
# output_csv = str(Path(fname).with_suffix('.csv'))
# with open(fname, 'r', encoding='utf-8') as txtfile, \
#      open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
#     reader = csv.reader(txtfile, delimiter=',')
#     writer = csv.writer(csvfile)
#     for row in reader:
#         writer.writerow(row)


In [ ]:
# graylock = GraylogAPIProcessor()
# fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra")/ f"{today_str}_ruOperation.txt"
# response = graylock.saveResponse(fname,"30-06-2025","01-07-2025",[],"ruOperation")
# output_csv = str(Path(fname).with_suffix('.csv'))
# csv.field_size_limit(10**6) 
# with open(fname, 'r', encoding='utf-8') as txtfile, \
#      open(output_csv, 'w', newline='', encoding='utf-8') as csvfile:
#     reader = csv.reader(txtfile, delimiter=',')
#     writer = csv.writer(csvfile)
#     for row in reader:
#         writer.writerow([str(cell) for cell in row])

In [ ]:
def loadRUOperationRequest(logs: list[str]):
        cambios = [el for el in logs if "ruOperationRequest" in el]
        xmls = f"<xml>{''.join(cambios)}</xml>"
        df_cambios = pd.read_xml(StringIO(xmls))

        rename_cols = {
            "runningDate": "FechaOrigen",
            "runningNumber": "NTécnico",
            "startLocation": "CódigoInicio",
            "startLocationSequence":"SecuenciaInicio",
            "startBookedTime": "HoraPlanificadaInicio",
            "enabled": "Activo",
            "operation": "Operación",
            "reason": "Razón",
            # "requestedLapse",
            "timestamp": "Fecha",
            "esbtimestamp": "FechaESB",
            # "endLocation": "CódigoFin",
            #  "endLocationSequence":"SecuenciaFin",
            # "endBookedTime":"HoraPlanificadaFin",
        }

        df_cambios = df_cambios[list(rename_cols.keys())].rename(columns=rename_cols)
        # df_cambios[["NTécnico", "CódigoInicio", "CódigoFin"]] = df_cambios[
        #     ["NTécnico", "CódigoInicio", "CódigoFin"]
        # ].map(rellenarId)
        df_cambios[["Fecha", "FechaESB"]] = localizeFecha(
            df_cambios, ["Fecha", "FechaESB"]
        )
        # .transform(
        #     lambda x: x.dt.tz_localize("Europe/Madrid").dt.tz_convert(None), axis=0
        # )
        df_cambios["FechaOrigen"] = (
            df_cambios["FechaOrigen"]
            .astype(str)
            .apply(
                lambda x: (
                    "".join(regex.findall(r"\d+", x))[:8] if not isEmpty(x) else None
                )
            )
        )
        df_cambios[["FechaOrigen"]] = localizeFecha(
            df_cambios, ["FechaOrigen"], format="%Y%m%d"
        )
        df_cambios["FechaOrigen"] = df_cambios["FechaOrigen"].dt.date
        df_cambios: pd.DataFrame = df_cambios.drop_duplicates().sort_values(
            by=["Fecha"]
        )
        # df_cambios[["NombreInicio", "NombreFin"]] = df_cambios[
        #     ["CódigoInicio", "CódigoFin"]
        # ].map(self.map_codigo_estacion.get)
    

        return df_cambios

In [ ]:
sitraproceso = SitraProcessor()
# fname_sitra=Path(r"c:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra")/ f"{today_str}_sitra.csv"
# fname_ru = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\graylog\sitra")/ f"{today_str}_ruOperation.csv"
fname_sitra = Path(r"c:\Users\xiangzhou.zhang\Downloads\All-Messages-search-result (13).csv")
logs_ru = sitraproceso.readLogFile(fname_sitra)
df_ru = loadRUOperationRequest(logs_ru)
log_rm = sitraproceso.readLogFile(fname_sitra)
df_rm = sitraproceso.loadRealMovement(log_rm)
# df_rp = sitraproceso.loadRealParking(log_rm)


In [ ]:
df_ru

In [ ]:

df_ru[df_ru["NTécnico"]==52728]

In [ ]:
# data ={
#     "realMovement":df_rm,
#     "realParking":df_rp
# }
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\Ru_.xlsx")
guardarExcel(df_ru,fname)

In [ ]:
fname = Path

In [ ]:
conteo = df_rm.groupby(["Fuente","Código"]).size().reset_index(name="Conteo")

In [ ]:
K = conteo[conteo["Fuente"] == "STACrail"]

In [ ]:
K.head(5)

In [ ]:
estaciones_con_ctc = loadEstaciones()

In [ ]:
estaciones_sin_ctc = loadEstacionSinCTC()

In [ ]:
estaciones = pd.concat([estaciones_con_ctc, estaciones_sin_ctc], ignore_index=True)

In [ ]:
estaciones[estaciones["Código"] == "05505"]

In [ ]:
K_count = pd.merge(
    K,
    estaciones[["Delegación","Código","Nombre"]],
    on="Código",
    how = "left"
)

In [ ]:
K_count =K_count[["Delegación","Código","Nombre","Conteo"]] 

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\rm_sin_ctc\2025-10-23_K_conteo.xlsx")

In [ ]:
guardarExcel(K_count,fname)

In [ ]:
strail = df_rm[df_rm["Fuente"] =="STACrail"]

In [ ]:
strail[strail["Código"] =="14223"]

In [ ]:

INFIESTO_UNQUERA = [
    "05535", "05537", "05539", "05541", "05542", "05543", "05545", "05547", "05549", "05551",
    "05553", "05555", "05557", "05559", "05561", "05563", "05565", "05567", "05569", "05571",
    "05573", "05575", "05577", "05579", "05649"
]


In [ ]:
RIBADEO_VILLADEMAR =  [
    "05141", "05143", "05145", "05147", "05149", "05151", "05153", "05155", "05156", "05157",
    "05159", "05161", "05163", "05165", "05167", "05169", "05171", "05173", "05175", "05177",
    "05179", "05181", "05183", "05185", "05187", "05291", "05289", "05193", "05197", "05199",
    "05299", "05297", "05295", "05293", "05191", "05189", "05287", "05285", "05283", "05281",
    "05279", "05277", "05275", "05273", "05271", "05269", "05267", "05265", "05263", "05261",
    "05259", "05257", "05255", "05253", "05251", "05249", "05247"
]

In [ ]:
duplicados = [item for item, count in Counter(RIBADEO_VILLADEMAR).items() if count > 1]
print(duplicados)  

In [ ]:
INFIESTO = df_rm[df_rm["Código"].isin(INFIESTO_UNQUERA)].copy()

In [ ]:
INFIESTO["Código"] = pd.Categorical(INFIESTO["Código"],categories=INFIESTO_UNQUERA,ordered=True)

In [ ]:
INFIESTO.sort_values("Código",inplace= True)

In [ ]:
RIBADEO = df_rm[df_rm["Código"].isin(RIBADEO_VILLADEMAR)]

In [ ]:
RIBADEO[RIBADEO["Código"].duplicated()]

In [ ]:
RIBADEO["Código"] = pd.Categorical(RIBADEO["Código"],categories=RIBADEO_VILLADEMAR,ordered=True)

In [ ]:
RIBADEO.sort_values("Código",inplace= True)

In [ ]:
ayer = datetime.now() - timedelta(days=1)
ayer_str = ayer.strftime("%Y-%m-%d")
INFIESTO["Código"] = INFIESTO["Código"].astype(str)
RIBADEO["Código"] = RIBADEO["Código"].astype(str)
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe\rm_sin_ctc") / f"{ayer_str}rm_estaciones_sin_ctc.xlsx"
data ={
    "INFIESTO_UNQUERA": INFIESTO,
    "RIBADEO_VILLADEMAR": RIBADEO,
    # "RU_Operation_Request": df_ru,
}

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
df_rm["FechaOrigen"] = pd.to_datetime(df_rm["FechaOrigen"])
df_rm = df_rm[df_rm["FechaOrigen"] == pd.to_datetime("2025-06-30")]

In [ ]:
df_rp["FechaOrigen"] = pd.to_datetime(df_rp["FechaOrigen"])
df_rp = df_rp[df_rp["FechaOrigen"] == pd.to_datetime("2025-06-30")]

In [ ]:
df_ru["FechaOrigen"] = pd.to_datetime(df_ru["FechaOrigen"])
df_ru = df_ru[df_ru["FechaOrigen"] == pd.to_datetime("2025-06-30")]

In [ ]:
df_rm1= df_rm[df_rm["Código"] == "78400"].copy()
df_rp1 = df_rp[df_rp["Código"] == "78400"].copy()

In [ ]:
# df_rm1["FechaOrigen"] = df_rm1["FechaOrigen"].astype(str)
# df_rp1["FechaOrigen"] = df_rp1["FechaOrigen"].astype(str)
# df_ru["FechaOrigen"] = df_ru["FechaOrigen"].astype(str)

data= {
    # "RealMovement":df_rm1,
    # "RealParking": df_rp1,
    "RuOperation": df_ru
}

In [ ]:
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\ru_29+.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
df_rp_13900 = df_rp[df_rp["NTécnico"] == "13900"]

In [ ]:
df_rp_13900

In [ ]:
df_rp_13900 = df_rp_13900[df_rp_13900["Código"] == "01007"]

In [ ]:
df_rp_13900

In [ ]:
df_rm_13900 = df_rm[df_rm["NTécnico"] == "13900"]

In [ ]:
df_rm_13900


In [ ]:
df_rm_13900 = df_rm_13900[df_rm_13900["Código"] == "01007"]

In [ ]:
df_rm_13900

In [ ]:
data= {
    "RealMovement":df_rm_13900,
    "RealParking": df_rp_13900,
}

In [ ]:
fname =Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\13900_sitra.xlsx")

In [ ]:
guardarExcelMulti(data,fname)

In [ ]:
df_rm.sort_values(by=["FechaOrigen"])

In [ ]:
df_stop=df_ru[df_ru['Operación'] == 'stopLapse']

In [ ]:
df_IO = df_ru[df_ru['Operación'] == 'intermediateOriginByIncidence'].copy()

In [ ]:
df_rs = df_rm[(df_rm["CodigoRetrasoSalida"] == "DET") | (df_rm["CodigoRetrasoSalida"] == "DPK")].copy()

In [ ]:
df_rl = df_rm[(df_rm["CodigoRetrasoLlegada"] == "DET") | (df_rm["CodigoRetrasoLlegada"] == "DPK")].copy()

In [ ]:
today_str = (datetime.now() - timedelta(days=1)).strftime("%Y%m%d")
df_SL = pd.DataFrame(df_stop)
df_SL["Activo"] = df_SL["Activo"].astype(str)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual")/ f"{today_str}_ruOperation.xlsx"

In [ ]:
data={"StopLapse":df_SL,"intermediateOriginByIncidence":df_IO}
guardarExcelMulti(data, fname)

In [ ]:
df_SL = pd.DataFrame(df_stop)

In [ ]:
fname = Path(r"C:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual")/ f"{today_str}_DPK_DET.xlsx"

In [ ]:
data={"LLEGADA":df_rl,"SALIDA":df_rs}
guardarExcelMulti(data, fname)


<h1> RealMovement <h1>

In [ ]:
df_automatico = df_rm[df_rm["Fuente"] =="Automático"]


In [ ]:
estaciones = loadEstaciones()
sinCTC = loadEstacionSinCTC()

In [ ]:
estaciones = estaciones[["Código","Nombre"]]
sinCTC = sinCTC[["Código","Nombre"]]

In [ ]:
estaciones[estaciones["Código"]=="31400"]

In [ ]:
df_automatico_conteo = df_automatico.groupby(["Código","Movimiento"]).size().reset_index(name="conteo")

In [ ]:
df_automatico_conteo = pd.merge(
    estaciones,
    df_automatico_conteo,
    how ="right",
    on = ["Código"]
)

In [ ]:
df_automatico_conteo.loc[df_automatico_conteo["Nombre"].isna(), "CTC"] = False
df_automatico_conteo.loc[df_automatico_conteo["CTC"].isna(), "CTC"] = True

In [ ]:
df_automatico_conteo = pd.merge(
    sinCTC,
    df_automatico_conteo,
    how ="inner",
    on = ["Código"]
)

In [ ]:
df_automatico_conteo.drop(columns=["Nombre_y"],inplace=True)
df_automatico_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)


In [ ]:
df_manual = df_rm[df_rm["Fuente"] == "Manual"]

In [ ]:
df_manual_conteo = df_manual.groupby(["Código","Movimiento"]).size().reset_index(name="conteo")

In [ ]:
df_manual_conteo = pd.merge(
    estaciones,
    df_manual_conteo,
    how ="right",
    on = ["Código"]
)

In [ ]:
df_manual_conteo.loc[df_manual_conteo["Nombre"].isna(), "CTC"] = False
df_manual_conteo.loc[df_manual_conteo["CTC"].isna(), "CTC"] = True
df_manual_conteo = pd.merge(
    sinCTC,
    df_manual_conteo,
    how ="inner",
    on = ["Código"]
)
df_manual_conteo.drop(columns=["Nombre_y"],inplace=True)
df_manual_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_Davinci = df_rm[df_rm["Fuente"] == "Davinci"]

In [ ]:
df_Davinci_conteo = df_Davinci.groupby(["Código","Movimiento"]).size().reset_index(name="conteo")

In [ ]:
df_Davinci_conteo = pd.merge(
    estaciones,
    df_Davinci_conteo,
    how ="right",
    on = ["Código"]
)


In [ ]:
df_Davinci_conteo.loc[df_Davinci_conteo["Nombre"].isna(), "CTC"] = False
df_Davinci_conteo.loc[df_Davinci_conteo["CTC"].isna(), "CTC"] = True
df_Davinci_conteo = pd.merge(
    sinCTC,
    df_Davinci_conteo,
    how ="inner",
    on = ["Código"]
)
df_Davinci_conteo.drop(columns=["Nombre_y"],inplace=True)
df_Davinci_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_k = df_rm[df_rm["Fuente"] == "STACrail"]

In [ ]:
df_k_conteo = df_k.groupby(["Código","Movimiento"]).size().reset_index(name="conteo")

In [ ]:
df_k_conteo = pd.merge(
    estaciones,
    df_k_conteo,
    how ="right",
    on = ["Código"]
)


In [ ]:
df_k_conteo.loc[df_k_conteo["Nombre"].isna(), "CTC"] = False
df_k_conteo.loc[df_k_conteo["CTC"].isna(), "CTC"] = True
df_k_conteo = pd.merge(
    sinCTC,
    df_k_conteo,
    how ="inner",
    on = ["Código"]
)
df_k_conteo.drop(columns=["Nombre_y"],inplace=True)
df_k_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_automatico_conteo = df_automatico_conteo.rename(columns={
 "conteo": "Conteo_auto"
})
df_Davinci_conteo = df_Davinci_conteo.rename(columns={
     "conteo": "Conteo_Davinci"
})
df_k_conteo = df_k_conteo.rename(columns={
     "conteo": "Conteo_k"
})
df_manual_conteo = df_manual_conteo.rename(columns={
     "conteo": "Conteo_manual"
})


est = df_automatico_conteo.merge(df_Davinci_conteo[["Código","Nombre","Movimiento","Conteo_Davinci","CTC"]], on=["Código","Movimiento","Nombre","CTC"], how="outer") \
    .merge(df_k_conteo[["Código","Nombre","Movimiento","Conteo_k","CTC"]], on=["Código","Movimiento","Nombre","CTC"], how="outer") \
    .merge(df_manual_conteo[["Código","Nombre","Movimiento","Conteo_manual","CTC"]], on=["Código","Movimiento","Nombre","CTC"], how="outer")


In [ ]:
est

In [ ]:
rm = est[["Código","Nombre","Movimiento","CTC","Conteo_auto","Conteo_Davinci","Conteo_k","Conteo_manual"]]

In [ ]:
rm

<h1>RealParkingTrack</h1>


In [ ]:
df_ager = df_rp[df_rp["Fuente"] == "Ager"]
df_CTC = df_rp[df_rp["Fuente"] == "CTC"]
df_Sitra = df_rp[df_rp["Fuente"] == "Sitra"]
df_planif = df_rp[df_rp["Fuente"] == "Planif"]

In [ ]:
df_ager_conteo = df_ager.groupby(["Código","Asignacion"]).size().reset_index(name="conteo")
df_CTC_conteo = df_CTC.groupby(["Código","Asignacion"]).size().reset_index(name="conteo")
df_Sitra_conteo = df_Sitra.groupby(["Código","Asignacion"]).size().reset_index(name="conteo")
df_planif_conteo = df_planif.groupby(["Código","Asignacion"]).size().reset_index(name="conteo")

In [ ]:
df_ager_conteo = pd.merge(
    estaciones,
    df_ager_conteo,
    how ="right",
    on = ["Código"]
)
df_CTC_conteo = pd.merge(
    estaciones,
    df_CTC_conteo,
    how ="right",
    on = ["Código"]
)

df_Sitra_conteo = pd.merge(
    estaciones,
    df_Sitra_conteo,
    how ="right",
    on = ["Código"]
)

df_planif_conteo = pd.merge(
    estaciones,
    df_planif_conteo,
    how ="right",
    on = ["Código"]
)


In [ ]:
df_ager_conteo.loc[df_ager_conteo["Nombre"].isna(), "CTC"] = False
df_ager_conteo.loc[df_ager_conteo["CTC"].isna(), "CTC"] = True
df_ager_conteo = pd.merge(
    sinCTC,
    df_ager_conteo,
    how ="inner",
    on = ["Código"]
)
df_ager_conteo.drop(columns=["Nombre_y"],inplace=True)
df_ager_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_CTC_conteo.loc[df_CTC_conteo["Nombre"].isna(), "CTC"] = False
df_CTC_conteo.loc[df_CTC_conteo["CTC"].isna(), "CTC"] = True
df_CTC_conteo = pd.merge(
    sinCTC,
    df_CTC_conteo,
    how ="inner",
    on = ["Código"]
)
df_CTC_conteo.drop(columns=["Nombre_y"],inplace=True)
df_CTC_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_Sitra_conteo.loc[df_Sitra_conteo["Nombre"].isna(), "CTC"] = False
df_Sitra_conteo.loc[df_Sitra_conteo["CTC"].isna(), "CTC"] = True
df_Sitra_conteo = pd.merge(
    sinCTC,
    df_Sitra_conteo,
    how ="inner",
    on = ["Código"]
)
df_Sitra_conteo.drop(columns=["Nombre_y"],inplace=True)
df_Sitra_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_planif_conteo.loc[df_planif_conteo["Nombre"].isna(), "CTC"] = False
df_planif_conteo.loc[df_planif_conteo["CTC"].isna(), "CTC"] = True
df_planif_conteo = pd.merge(
    sinCTC,
    df_planif_conteo,
    how ="inner",
    on = ["Código"]
)
df_planif_conteo.drop(columns=["Nombre_y"],inplace=True)
df_planif_conteo.rename(columns={"Nombre_x":"Nombre"},inplace=True)

In [ ]:
df_ager_conteo = df_ager_conteo.rename(columns={
 "conteo": "Conteo_ager"
})
df_CTC_conteo = df_CTC_conteo.rename(columns={
     "conteo": "Conteo_CTC"
})
df_Sitra_conteo = df_Sitra_conteo.rename(columns={
     "conteo": "Conteo_Sitra"
})
df_planif_conteo = df_planif_conteo.rename(columns={
     "conteo": "Conteo_planif"
})


rp = df_ager_conteo.merge(df_CTC_conteo[["Código","Nombre","Asignacion","Conteo_CTC","CTC"]], on=["Código","Asignacion","Nombre","CTC"], how="outer") \
    .merge(df_Sitra_conteo[["Código","Nombre","Asignacion","Conteo_Sitra","CTC"]], on=["Código","Asignacion","Nombre","CTC"], how="outer") \
    .merge(df_planif_conteo[["Código","Nombre","Asignacion","Conteo_planif","CTC"]], on=["Código","Asignacion","Nombre","CTC"], how="outer")


In [ ]:
rp

In [ ]:
df_ager_conteo

In [ ]:
rp =rp[["Código","Nombre","Asignacion","CTC","Conteo_ager","Conteo_CTC","Conteo_Sitra","Conteo_planif"]]

In [ ]:
rm["CTC"] = rm["CTC"].astype(str)
rp["CTC"] = rp["CTC"].astype(str)
data={
    "RealMovement":rm,
    "RealParking": rp
}
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe\conteo")/ f"{today_str}_ConteoRP_RM.xlsx"
guardarExcelMulti(data,fname)

In [ ]:
df_ru

In [ ]:
tren_23811 = df_ru [df_ru["NTécnico"] == "23811"].copy()

In [ ]:
tren_23359_o = df_ru [df_ru["NTécnico"] == "23359"].copy()

In [ ]:
tren_23359_o

In [ ]:
tren_23707 = df_ru [df_ru["NTécnico"] == "23707"].copy()

In [ ]:
tren_23711 = df_ru [df_ru["NTécnico"] == "23711"].copy()

In [ ]:
tren_26516 = df_ru [df_ru["NTécnico"] == "26516"].copy()

In [ ]:
df_ru [df_ru["N"] == "26516"]



In [ ]:
tren_originbyincidence = df_ru[df_ru["Operación"] == "intermediateOriginByIncidence"].copy()

In [ ]:
tren__00752 = df_ru [df_ru["NTécnico"] == "00752"].copy()

In [ ]:
tren_00802 = df_ru [df_ru["NTécnico"] == "00802"].copy()


In [ ]:
tren_21607 = df_ru [df_ru["NTécnico"] == "20425"].copy()

In [ ]:
tren_21607

In [ ]:
df_rm 

In [ ]:
tren_21607 = df_rm[df_rm["NTécnico"] == "21607"].copy()

In [ ]:
tren_21607 = tren_21607[tren_21607["Movimiento"] == "SALIDA Guadiana"]

In [ ]:
tren_20425 = df_rm[(df_rm["NTécnico"] == "20425")  & (df_rm["Movimiento"] == "SALIDA Guadiana")].copy()

In [ ]:
tren_21609 = df_rm[(df_rm["NTécnico"] == "21609")  & (df_rm["Movimiento"] == "LLEGADA Guadiana")].copy()

In [ ]:
tren_23359 = df_rm[(df_rm["NTécnico"] == "23359")  & (df_rm["Movimiento"] == "LLEGADA Guadiana")].copy()

In [ ]:
tren_23359

In [ ]:
tren_23735 = df_rm[(df_rm["NTécnico"] == "23735")  & (df_rm["Movimiento"] == "SUPRESIÓN")].copy()

In [ ]:
tren_23735 = tren_23735[tren_23735["Fecha"] == "2026-01-14 08:22:55"]

In [ ]:
tren_23735


In [ ]:
tren_22321 = df_rm[(df_rm["NTécnico"] == "22321")  & (df_rm["Movimiento"] == "SUPRESIÓN")].copy()

In [ ]:
tren_22321 = tren_22321[tren_22321["Fecha"] == "2026-01-14 08:24:23"]

In [ ]:
fname = Path(r"c:\Users\xiangzhou.zhang\Downloads\All-Messages-search-result (9).csv")

In [ ]:
with fname.open("r", encoding="utf8") as f:
    data = f.read()
    if regex.search('^"timestamp', data):
        data = data.split("\n", 1)[-1]  
lines = removeDoubleQuotes(data)
lines


In [ ]:
from src.processor.xsiv_processor import XSIVProcessor

train_types = {
    "Approach": "APROXIMACIÓN",
    "Arrival": "LLEGADA",
    "Departure": "SALIDA",
    "Elimination": "BAJA",
    "End": "FIN",
    # "Entry": "ENTRY",
    # "Exit": "EXIT",
    "Maneuver": "MANIOBRA",
    "Platform": "ALTA",
    "PlatformForecast": "PREVISIÓN",  # "PREDICCIÓN",
    "Stopped": "STOP",
    # "TrackingLost": "LOST_TRACK",
}
xsivProcesor = XSIVProcessor()
logs = xsivProcesor.readLogFile(fname, train_types=train_types, train_operator=None)
logs

In [ ]:
def processLogInfo(log: str):
    """
    Procesa el contenido del log
    """
    info = {}
    mse_xsiv = ET.fromstring(log)
    info.update(dict(mse_xsiv.items()))
    info["movementType"] = mse_xsiv[0].tag
    info.update(dict(mse_xsiv[0].items()))
    for el in mse_xsiv[0]: 
        if el.text:
            info[el.tag] = el.text
        info.update({f"{el.tag}_{k}": v for k, v in el.items()})
    return info

In [ ]:
import xml.etree.ElementTree as ET

def procesar_logs_con_registertype(logs):
    """
    Convierte cada línea de logs en XML válido con registerType dentro del nodo raíz.
    
    Args:
        logs (list[str]): Lista de líneas de log CSV con campo XML y registerType al final.
    
    Returns:
        list[str]: Lista de strings XML válidos con registerType incluido.
    """
    xml_limpios = []

    for log in logs:
        try:
            # Separar XML del registerType
            if '","' in log:
                xml_part, register_type = log.split('","', 1)
                register_type = register_type.strip()
            else:
                xml_part = log
                register_type = ""

            # Asegurar cierre del XML
            if "</mse-xsiv>" in xml_part:
                xml_part = xml_part.split("</mse-xsiv>")[0] + "</mse-xsiv>"

            # Parsear XML
            root = ET.fromstring(xml_part.encode("utf-8"))

            # Añadir registerType al nodo raíz
            if register_type:
                root.set("registerType", register_type)

            # Convertir a string XML limpio
            xml_limpios.append(ET.tostring(root, encoding="unicode"))

        except ET.ParseError as e:
            print(f"⚠️ Error al procesar log: {e}")
            continue

    return xml_limpios


In [ ]:
logs_procesado = procesar_logs_con_registertype(logs)

In [ ]:
data = parallelizeFunction(processLogInfo, logs_procesado, show_progress=False)
data

In [ ]:
df_xsiv = pd.DataFrame(data)

In [ ]:
df_xsiv.columns

In [ ]:
tren_63100 = df_xsiv[df_xsiv["trainCode"] == "63100"].copy()

In [ ]:
tren_63100 = tren_63100[tren_63100["controlPoint_sequence"] == "94"]

In [ ]:
tren_90225 = df_xsiv[df_xsiv["trainCode"] == "90225"].copy()


In [ ]:
tren_90225 = tren_90225[tren_90225["timestamp"] == "2026-01-14T07:04:18.586+01:00"]

In [ ]:
tren_71800 = df_xsiv[df_xsiv["trainCode"] == "71800"].copy()

In [ ]:
tren_21869 = df_xsiv[df_xsiv["trainCode"] == "21869"].copy()

In [ ]:
origen_change = pd.concat([tren_23811, tren_23359_o],ignore_index=True)

In [ ]:
destination_change = pd.concat([tren_23707, tren_23711 ],ignore_index=True)

In [ ]:
destination_change 


In [ ]:
interrupcion = tren_26516.copy()

In [ ]:
origen_by_incidence = tren_originbyincidence.copy()

In [ ]:
stop_lapse = pd.concat([tren__00752, tren_00802], ignore_index=True)

In [ ]:
Salida_Guadiana = pd.concat([tren_21607,tren_20425], ignore_index=True)

In [ ]:
Llegada_Guadiana = pd.concat([tren_21609,tren_23359], ignore_index=True)

In [ ]:
Denteción_entrada = pd.concat([tren_63100, tren_90225], ignore_index=True)

In [ ]:
Detención_trayecto = pd.concat([tren_71800, tren_21869], ignore_index=True)

In [ ]:
Supresión = pd.concat([tren_23735, tren_22321], ignore_index=True)

In [ ]:
data = {
    "Origen_by_incidence": origen_by_incidence,
    "Stop_lapse": stop_lapse,
    "Salida_Guadiana": Salida_Guadiana,
    "Llegada_Guadiana": Llegada_Guadiana,
    "Denteción_entrada": Denteción_entrada,
    "Detención_trayecto": Detención_trayecto,
    "Supresión": Supresión
}
fname = Path(r"c:\Users\xiangzhou.zhang\Documents\Data\Informe_puntual\CirculationState.xlsx")

In [ ]:
guardarExcelMulti(data,fname)